# Stage-3 Router — training + Lever 2 (Colab CPU)

Trains the question-conditioned router and tests **why the question-only router failed**
by comparing three input modes:

| mode | router input |
|------|--------------|
| **text** | question embedding only (original pilot) |
| **visual** | pooled image features only (global + gaze-crop CLIP) |
| **text+visual** | both (image-aware router — Lever 2) |

If detail-need is image-dependent, **text+visual** should lift PR-AUC above the
text-only 0.207. Feasibility pilot on the 300-sample run (SmolVLM2-256M); **CPU only**.
Reads the WearVQA images from Drive (`/content/drive/MyDrive/wearvqa_gaze_only`).

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers scikit-learn matplotlib

## 2. Clone repo & enter the router folder

In [ ]:
import os
REPO = 'https://github.com/shubhamOjha1000/AAAI_2027_code.git'
if not os.path.isdir('/content/AAAI_2027_code'):
    !git clone "$REPO" /content/AAAI_2027_code
else:
    !cd /content/AAAI_2027_code && git pull --ff-only
%cd /content/AAAI_2027_code/stage3_router
!ls -la

## 3. Mount Drive (for the WearVQA images)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/wearvqa_gaze_only | head

## 4. Build labels  (DETAIL iff global wrong & fovea right)

In [ ]:
!python build_router_labels.py

## 5. Encode questions  (sentence-transformers, CPU)

In [ ]:
!python extract_question_features.py

## 6. Encode images  (CLIP: global + gaze-crop, CPU) — Lever 2 features

In [ ]:
!python extract_visual_features.py

## 7. Stratified + grouped 5-fold splits

In [ ]:
!python splits.py

## 8. Train + evaluate all three feature modes

In [ ]:
import os, subprocess
for mode in ["text", "visual", "text+visual"]:
    print(f"\n========== MODE: {mode} ==========")
    env = dict(os.environ, FEATURE_MODE=mode)
    subprocess.run(["python", "train_router.py"], env=env, check=True)
    subprocess.run(["python", "eval_router.py"], env=env, check=True)

## 9. Compare the three modes

In [ ]:
import json
tags = {"text": "text", "visual": "visual", "text+visual": "text_visual"}
print(f"{'mode':12s} {'PR_AUC':>7} {'DET_F1':>7}   router_learned [acc%, tokens]")
print("-" * 56)
for mode, tag in tags.items():
    m = json.load(open(f"outputs/metrics_{tag}.json"))
    r, e = m["router"], m["end_to_end"]
    print(f"{mode:12s} {r['PR_AUC']:7.3f} {r['DETAIL_f1']:7.3f}   {e['router_learned']}")
e = json.load(open("outputs/metrics_text.json"))["end_to_end"]
print("\nanchors:")
for k in ["oracle_maxBC", "type_prior", "always_full_A", "always_fovea_C", "always_global_B"]:
    print(f"  {k:18s} {e[k]}")
print("\n(text-only baseline PR-AUC was 0.207 -> does visual lift it?)")

## 10. Pareto plots (per mode)

In [ ]:
from IPython.display import Image, display
for tag in ["text", "visual", "text_visual"]:
    p = f"outputs/pareto_{tag}.png"
    print(p); display(Image(p))